In [ ]:
import torch
from lieflow.groups import Aff2
import matplotlib.pyplot as plt

In [ ]:
aff2 = Aff2()

In [ ]:
shape_1 = (100, 1, 5)
shape_2 = (4, 5)

g_1 = torch.zeros(*shape_1, 6)
g_1[..., :2] = torch.randn(*shape_1, 2)
r = (torch.rand(*shape_1) - 0.5) * torch.pi
cos, sin = r.cos(), r.sin()
R = torch.stack([cos, -sin, sin, cos], dim=-1).view(*shape_1, 2, 2)
s = torch.randn(*shape_1, 3)
S = torch.matrix_exp(torch.stack([s[..., 0], s[..., 2], s[..., 2], s[..., 1]], dim=-1).view(*shape_1, 2, 2))
g_1[..., 2:] = (R @ S).flatten(start_dim=-2, end_dim=-1)

g_2 = torch.zeros(*shape_2, 6)
g_2[..., :2] = torch.randn(*shape_2, 2)
r = (torch.rand(*shape_2) - 0.5) * torch.pi
cos, sin = r.cos(), r.sin()
R = torch.stack([cos, -sin, sin, cos], dim=-1).view(*shape_2, 2, 2)
s = torch.randn(*shape_2, 3)
S = torch.matrix_exp(torch.stack([s[..., 0], s[..., 2], s[..., 2], s[..., 1]], dim=-1).view(*shape_2, 2, 2))
g_2[..., 2:] = (R @ S).flatten(start_dim=-2, end_dim=-1)

e = torch.tensor([0., 0., 1., 0., 0., 1.])

(e - aff2.L(g_1, aff2.L_inv(g_1, e))).abs().max()

In [ ]:
g_1 = torch.zeros(*shape_1, 6)
g_1[..., :2] = torch.randn(*shape_1, 2)
r = (torch.rand(*shape_1) - 0.5) * torch.pi
cos, sin = r.cos(), r.sin()
R = torch.stack([cos, -sin, sin, cos], dim=-1).view(*shape_1, 2, 2)
s = torch.randn(*shape_1, 3)
S = torch.matrix_exp(torch.stack([s[..., 0], s[..., 2], s[..., 2], s[..., 1]], dim=-1).view(*shape_1, 2, 2))
g_1[..., 2:] = (R @ S).flatten(start_dim=-2, end_dim=-1)

In [ ]:
N = 2**14
g = torch.zeros(N, 6)
g[..., :2] = torch.randn(N, 2)
r = (torch.rand(N) - 0.5) * torch.pi
cos, sin = r.cos(), r.sin()
R = torch.stack([cos, -sin, sin, cos], dim=-1).view(N, 2, 2)
s = torch.randn(N, 3)
S = torch.matrix_exp(torch.stack([s[..., 0], s[..., 2], s[..., 2], s[..., 1]], dim=-1).view(N, 2, 2))
g[..., 2:] = (R @ S).flatten(start_dim=-2, end_dim=-1)
A = torch.randn(N, 6)
fig, ax = plt.subplots(1, 2, layout="constrained")
log_after_exp = (aff2.log(aff2.exp(A)) - A).abs().flatten()
ax[0].hist(log_after_exp)
ax[0].set_title("log after exp")
exp_after_log = (aff2.exp(aff2.log(g)) - g).abs().flatten()
ax[1].hist(exp_after_log)
ax[1].set_title("exp after log")
threshold = 1e-2
print(f"log after exp: {(100. * (log_after_exp > threshold).sum() / N).item():.2f} %")
print(f"exp after log: {(100. * (exp_after_log > threshold).sum() / N).item():.2f} %")

In [ ]:
def acts(g, point, frame):
    t = g[..., :2]
    A = g[..., 2:].view(*g.shape[:-1], 2, 2)
    return t + (A @ point[..., None]).squeeze(-1), A @ frame

In [ ]:
g_1 = torch.zeros(1, 6)
g_1[..., :2] = torch.randn(1, 2)
r = (torch.rand(1) - 0.5) * torch.pi
cos, sin = r.cos(), r.sin()
R = torch.stack([cos, -sin, sin, cos], dim=-1).view(1, 2, 2)
s = torch.randn(1, 3)
S = torch.matrix_exp(torch.stack([s[..., 0], s[..., 2] * 0.4, s[..., 2] * 0.4, s[..., 1]], dim=-1).view(1, 2, 2))
g_1[..., 2:] = (R @ S).flatten(start_dim=-2, end_dim=-1)

point_ref = torch.tensor([0., 0.])
frame_ref = torch.eye(2)

ts = torch.linspace(0., 1., 5)
A = aff2.log(g_1)
gs = aff2.exp(ts[..., None] * A)

points, frames = acts(gs, point_ref, frame_ref)

fig, ax = plt.subplots(1, 1)
ax.quiver(points[..., 0], points[..., 1], frames[..., 0, 0], frames[..., 1, 0], color="red", scale=8)
ax.quiver(points[..., 0], points[..., 1], frames[..., 0, 1], frames[..., 1, 1], color="blue", scale=8)
ax.quiver(points[0, 0], points[0, 1], frames[0, 0, 0], frames[0, 1, 0], color="darkred", scale=8)
ax.quiver(points[0, 0], points[0, 1], frames[0, 0, 1], frames[0, 1, 1], color="darkblue", scale=8)
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect("equal")